In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *

In [2]:
#files = sorted(glob.glob('/home/ulyanov/data/solo/phi/polar/*'))
files = sorted(glob.glob('/home/ulyanov/data/sdo/hmi/polar/*'))
print(len(files))
print(files[0], files[-1])

356
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_000000_TAI.3.magnetogram.fits /home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250530_220000_TAI.3.magnetogram.fits


In [6]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 512

view_north = View(nx, ny, xc, yc, Rsun, -90, -90, 0) ### North pole view
grid = np.mgrid[:nx,:ny].astype(np.float32)

mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

for file in files:
    print(file)

    with fits.open(file) as hdul:
        header = hdul[1].header.copy()
        data = hdul[1].data.copy()

    data = rebin(data, 4, update_header=header)
    view = View.from_header(header)

    transform = view_north.to_spherical(correct_mu=True, mu_thr=0.1) - view.to_spherical(correct_mu=True, correct_dr=True, mu_thr=0.1)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    n = (~np.isnan(data)) * (1 / alpha) ** 2
    coverage += np.nan_to_num(n)
    mean_ = mean.copy()
    mean += np.nan_to_num((data - mean) * n / coverage)
    variance += np.nan_to_num((data - mean) * (data - mean_) * n)

variance = variance / coverage
mean[coverage == 0] = np.nan
coverage[coverage == 0] = np.nan
variance[coverage == 0] = np.nan

/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_000000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_020000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_040000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_060000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_080000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_100000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_120000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_140000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_160000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_180000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_200000_TAI.3.magnetogram.fits
/home/ulyanov/data/sdo/hmi/polar/hmi.M_720s.20250501_220000_TAI.3.magnetogram.fits
/hom

/tmp/ipykernel_100252/3376200185.py:30: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [9]:
show_polar_data(mean, vmin=-20, vmax=20, cmap='seismic', label=r'$B_{los}, G$')

(<Figure size 1000x1000 with 2 Axes>, <Axes: >)

In [10]:
show_polar_data(np.sqrt(variance), vmin=0, vmax=20, cmap='turbo', label=r'$\sigma_B, G$')

/tmp/ipykernel_100252/3381950695.py:1: RuntimeWarning: invalid value encountered in sqrt
  show_polar_data(np.sqrt(variance), vmin=0, vmax=20, cmap='turbo', label=r'$\sigma_B, G$')


(<Figure size 1000x1000 with 2 Axes>, <Axes: >)